# ESCI reranker training

This notebook trains the cross-encoder reranker on ESCI (MSE loss on gains, nDCG eval).

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"

## Prepare train/test splits (run once)

Ensures `esci_train.parquet` and `esci_test.parquet` exist under `data/`.

In [ ]:
from src.data.load_data import load_esci
from src.data.utils import EXAMPLES_FILENAME

if (DATA_DIR / EXAMPLES_FILENAME).exists():
    raw_data_dir = DATA_DIR
else:
    raw_data_dir = None

df = load_esci(data_dir=raw_data_dir, save_splits_dir=DATA_DIR)
print(f"Loaded {len(df)} rows; splits written under {DATA_DIR}")

## Configure and run training

In [ ]:
import logging
from src.training.train_reranker import run_training

logging.basicConfig(level=logging.INFO, format="%(message)s")

train_cfg = {
    "data_dir": DATA_DIR,
    "model_name": "cross-encoder/ms-marco-MiniLM-L-12-v2",
    "product_col": "product_text",
    "save_path": DATA_DIR / "reranker",
    "epochs": 1,
    "batch_size": 32,
    "lr": 7e-6,
    "warmup_steps": 5000,
    "evaluation_steps": 5000,
}

model = run_training(**train_cfg)